# Engine (setup)

Shared inputs for every notebook in this folder: constants, the Liu tight-binding parameters, the Hamiltonians with spin-orbit coupling and the Zeeman field, the k-grids and their masks, the band quantities, the three field terms of the orbital moment, and the injection- and shift-current kernels. Nothing heavy runs here, so `%run engine.ipynb` takes a second. Each topic notebook starts with it and then computes its own figures.

| notebook | content |
|---|---|
| `nn_tnn.ipynb` | Figs. 1-4: bands, Berry curvature, quantum metric, Chern numbers |
| `response.ipynb` | Fig. 5: injection and shift current of the four materials |
| `kp.ipynb` | Table II ($\kappa$ by Löwdin reduction), Fig. 6, the first-principles k.p $\chi$ |
| `field.ipynb` | Figs. 7 and 8, and the decomposition into field terms |
| `edge.ipynb` | Figs. 9, 10 and 11 |
| `soc.ipynb` | Fig. 12 |

Results of the heavy cells are saved in `data/` and figures in `figures/`, both next to the notebooks. The run times quoted in the notebooks were measured on a 2-core machine.

## Imports

In [ ]:
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.ticker import MaxNLocator
from matplotlib.lines import Line2D

os.makedirs("data", exist_ok=True)
os.makedirs("figures", exist_ok=True)

## Publication styling

In [ ]:
mpl.rcParams.update({
    "font.family": "serif", "font.serif": ["cmr10", "DejaVu Serif"], "mathtext.fontset": "cm",
    "axes.formatter.use_mathtext": True, "axes.unicode_minus": False,
    "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8, "legend.fontsize": 7,
    "xtick.labelsize": 7, "ytick.labelsize": 7,
    "axes.linewidth": 0.6, "lines.linewidth": 1.1, "lines.markersize": 3.5,
    "xtick.direction": "in", "ytick.direction": "in", "xtick.top": True, "ytick.right": True,
    "xtick.major.size": 3, "ytick.major.size": 3, "xtick.minor.size": 1.6, "ytick.minor.size": 1.6,
    "xtick.minor.visible": True, "ytick.minor.visible": True,
    "legend.frameon": False, "legend.handlelength": 1.8,
    "savefig.dpi": 600, "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
})

TW, CW = 7.0, 3.4      # text and column width of the paper (inches)
BLUE, RED, ORANGE, GREY = "#0072B2", "#C0392B", "#E69F00", "0.55"
UNIT = r"\mathrm{A\,m\,V^{-2}\,s^{-1}}"
SUNIT = r"\mathrm{nm\,\mu A\,V^{-2}}"


def label(ax, s, x=0.0, y=1.02, va="bottom"):
    ax.text(x, y, r"$\mathbf{(%s)}$" % s, transform=ax.transAxes, fontsize=9, ha="left", va=va)


def zero(ax, v=False):
    ax.axhline(0, color="0.8", lw=0.5, zorder=0)
    if v:
        ax.axvline(0, color="0.8", lw=0.5, zorder=0)


def mirror(B, Y):
    # eta(-B) = -eta(B), so the B >= 0 data fill the whole field range
    return np.concatenate([-B[:0:-1], B]), np.concatenate([-Y[:0:-1], Y])


def save(fig, name):
    fig.savefig(f"figures/{name}.pdf")
    fig.savefig(f"figures/{name}.png", dpi=200)


def fmt(c):
    val = c if abs(c) > 1e-10 else 0.0
    err = abs(c - round(c)) * 100
    return f"{val:.6f} ({err:.2e}% error)"

## Constants

In [ ]:
mu_B = 5.7883818012e-5     # eV/T
g_s = 2
e = 1.602176634e-19        # C (also J per eV)
hbar = 1.054571817e-34     # J s
m_e = 9.1093837015e-31     # kg

B80 = 0.08 / mu_B          # mu_B B = 80 meV, the enlarged Zeeman field of Figs. 1-5

materials = ["MoS2", "MoSe2", "WS2", "WSe2"]
names = {"MoS2": r"MoS$_2$", "MoSe2": r"MoSe$_2$", "WS2": r"WS$_2$", "WSe2": r"WSe$_2$"}
colors = {"MoS2": "#0072B2", "MoSe2": "#D55E00", "WS2": "#009E73", "WSe2": "#CC79A7"}

## Tight-binding parameters

Liu *et al.*, PRB **88**, 085433 (2013), Tables II-IV, GGA and LDA fits: lattice constant $a$ in Å, on-site energies and hoppings in eV. The TNN model uses the $a$ of the NN table, and the Ising coupling $\lambda$ is the same for both fits.

In [ ]:
NN_keys = ["a", "epsilon1", "epsilon2", "t0", "t1", "t2", "t11", "t12", "t22"]
NN_table = {
    "GGA": {"MoS2":  [3.190, 1.046, 2.104, -0.184, 0.401, 0.507, 0.218, 0.338, 0.057],
            "MoSe2": [3.326, 0.919, 2.065, -0.188, 0.317, 0.456, 0.211, 0.290, 0.130],
            "WS2":   [3.191, 1.130, 2.275, -0.206, 0.567, 0.536, 0.286, 0.384, -0.061],
            "WSe2":  [3.325, 0.943, 2.179, -0.207, 0.457, 0.486, 0.263, 0.329, 0.034]},
    "LDA": {"MoS2":  [3.129, 1.238, 2.366, -0.218, 0.444, 0.533, 0.250, 0.360, 0.047],
            "MoSe2": [3.254, 1.001, 2.239, -0.222, 0.350, 0.488, 0.244, 0.314, 0.129],
            "WS2":   [3.132, 1.355, 2.569, -0.238, 0.626, 0.557, 0.324, 0.405, -0.076],
            "WSe2":  [3.253, 1.124, 2.447, -0.242, 0.506, 0.514, 0.305, 0.353, 0.025]},
}

TNN_keys = ["epsilon1", "epsilon2", "t0", "t1", "t2", "t11", "t12", "t22",
            "r0", "r1", "r2", "r11", "r12", "u0", "u1", "u2", "u11", "u12", "u22"]
TNN_table = {
    "GGA": {"MoS2":  [0.683, 1.707, -0.146, -0.114, 0.506, 0.085, 0.162, 0.073, 0.060, -0.236, 0.067, 0.016, 0.087, -0.038, 0.046, 0.001, 0.266, -0.176, -0.150],
            "MoSe2": [0.684, 1.546, -0.146, -0.130, 0.432, 0.144, 0.117, 0.075, 0.039, -0.209, 0.069, 0.052, 0.060, -0.042, 0.036, 0.008, 0.272, -0.172, -0.150],
            "WS2":   [0.717, 1.916, -0.152, -0.097, 0.590, 0.047, 0.178, 0.016, 0.069, -0.261, 0.107, -0.003, 0.109, -0.054, 0.045, 0.002, 0.325, -0.206, -0.163],
            "WSe2":  [0.728, 1.655, -0.146, -0.124, 0.507, 0.117, 0.127, 0.015, 0.036, -0.234, 0.107, 0.044, 0.075, -0.061, 0.032, 0.007, 0.329, -0.202, -0.164]},
    "LDA": {"MoS2":  [0.820, 1.931, -0.176, -0.101, 0.531, 0.084, 0.169, 0.070, 0.070, -0.252, 0.084, 0.019, 0.093, -0.043, 0.047, 0.005, 0.304, -0.192, -0.162],
            "MoSe2": [0.715, 1.687, -0.154, -0.134, 0.437, 0.124, 0.119, 0.072, 0.048, -0.248, 0.090, 0.066, 0.045, -0.067, 0.041, 0.005, 0.327, -0.194, -0.151],
            "WS2":   [0.905, 2.167, -0.175, -0.090, 0.611, 0.043, 0.181, 0.008, 0.075, -0.282, 0.127, 0.001, 0.114, -0.063, 0.047, 0.004, 0.374, -0.224, -0.177],
            "WSe2":  [0.860, 1.892, -0.152, -0.125, 0.508, 0.094, 0.129, 0.009, 0.044, -0.278, 0.129, 0.059, 0.058, -0.090, 0.039, 0.001, 0.392, -0.224, -0.165]},
}

lambda_SOC = {"MoS2": 0.073, "MoSe2": 0.091, "WS2": 0.211, "WSe2": 0.228}


def params(mat, func="GGA", model="TNN"):
    if model == "NN":
        return dict(zip(NN_keys, NN_table[func][mat]))
    p = dict(zip(TNN_keys, TNN_table[func][mat]))
    p["a"] = NN_table[func][mat][0]
    return p

## NN Hamiltonian

Basis $(d_{z^2}, d_{xy}, d_{x^2-y^2})$, $\alpha = k_x a/2$, $\beta = \sqrt{3}k_y a/2$. We set $a = 1$, so $k$ is in units of $1/a$ and velocities are in eV $a$.

In [ ]:
def H_NN(alpha, beta, p):
    epsilon1, epsilon2, t0, t1, t2, t11, t12, t22 = [p[k] for k in NN_keys[1:]]

    h0 = 2 * t0 * (np.cos(2 * alpha) + 2 * np.cos(alpha) * np.cos(beta)) + epsilon1
    h1 = -2 * np.sqrt(3) * t2 * np.sin(alpha) * np.sin(beta) + 2j * t1 * (np.sin(2 * alpha) + np.sin(alpha) * np.cos(beta))
    h2 = 2 * t2 * (np.cos(2 * alpha) - np.cos(alpha) * np.cos(beta)) + 2 * np.sqrt(3) * 1j * t1 * np.cos(alpha) * np.sin(beta)
    h11 = 2 * t11 * np.cos(2 * alpha) + (t11 + 3 * t22) * np.cos(alpha) * np.cos(beta) + epsilon2
    h22 = 2 * t22 * np.cos(2 * alpha) + (3 * t11 + t22) * np.cos(alpha) * np.cos(beta) + epsilon2
    h12 = np.sqrt(3) * (t22 - t11) * np.sin(alpha) * np.sin(beta) + 4j * t12 * np.sin(alpha) * (np.cos(alpha) - np.cos(beta))

    H = np.array([[h0, h1, h2], [np.conj(h1), h11, h12], [np.conj(h2), np.conj(h12), h22]])
    return np.moveaxis(H, (0, 1), (-2, -1))

## TNN Hamiltonian

In [ ]:
def H_TNN(alpha, beta, p):
    epsilon1, epsilon2, t0, t1, t2, t11, t12, t22 = [p[k] for k in TNN_keys[:8]]
    r0, r1, r2, r11, r12, u0, u1, u2, u11, u12, u22 = [p[k] for k in TNN_keys[8:]]

    V0 = epsilon1 + 2 * t0 * (2 * np.cos(alpha) * np.cos(beta) + np.cos(2 * alpha)) + 2 * r0 * (2 * np.cos(3 * alpha) * np.cos(beta) + np.cos(2 * beta)) + 2 * u0 * (2 * np.cos(2 * alpha) * np.cos(2 * beta) + np.cos(4 * alpha))

    ReV1 = -2 * np.sqrt(3) * t2 * np.sin(alpha) * np.sin(beta) + 2 * (r1 + r2) * np.sin(3 * alpha) * np.sin(beta) - 2 * np.sqrt(3) * u2 * np.sin(2 * alpha) * np.sin(2 * beta)
    ImV1 = 2 * t1 * np.sin(alpha) * (2 * np.cos(alpha) + np.cos(beta)) + 2 * (r1 - r2) * np.sin(3 * alpha) * np.cos(beta) + 2 * u1 * np.sin(2 * alpha) * (2 * np.cos(2 * alpha) + np.cos(2 * beta))
    V1 = ReV1 + 1j * ImV1

    ReV2 = 2 * t2 * (np.cos(2 * alpha) - np.cos(alpha) * np.cos(beta)) - (2 / np.sqrt(3)) * (r1 + r2) * (np.cos(3 * alpha) * np.cos(beta) - np.cos(2 * beta)) + 2 * u2 * (np.cos(4 * alpha) - np.cos(2 * alpha) * np.cos(2 * beta))
    ImV2 = 2 * np.sqrt(3) * t1 * np.cos(alpha) * np.sin(beta) + (2 / np.sqrt(3)) * np.sin(beta) * (r1 - r2) * (np.cos(3 * alpha) + 2 * np.cos(beta)) + 2 * np.sqrt(3) * u1 * np.cos(2 * alpha) * np.sin(2 * beta)
    V2 = ReV2 + 1j * ImV2

    V11 = epsilon2 + (t11 + 3 * t22) * np.cos(alpha) * np.cos(beta) + 2 * t11 * np.cos(2 * alpha) + 4 * r11 * np.cos(3 * alpha) * np.cos(beta) + 2 * (r11 + np.sqrt(3) * r12) * np.cos(2 * beta) + (u11 + 3 * u22) * np.cos(2 * alpha) * np.cos(2 * beta) + 2 * u11 * np.cos(4 * alpha)

    ReV12 = np.sqrt(3) * (t22 - t11) * np.sin(alpha) * np.sin(beta) + 4 * r12 * np.sin(3 * alpha) * np.sin(beta) + np.sqrt(3) * (u22 - u11) * np.sin(2 * alpha) * np.sin(2 * beta)
    ImV12 = 4 * t12 * np.sin(alpha) * (np.cos(alpha) - np.cos(beta)) + 4 * u12 * np.sin(2 * alpha) * (np.cos(2 * alpha) - np.cos(2 * beta))
    V12 = ReV12 + 1j * ImV12

    V22 = epsilon2 + (3 * t11 + t22) * np.cos(alpha) * np.cos(beta) + 2 * t22 * np.cos(2 * alpha) + 2 * r11 * (2 * np.cos(3 * alpha) * np.cos(beta) + np.cos(2 * beta)) + (2 / np.sqrt(3)) * r12 * (4 * np.cos(3 * alpha) * np.cos(beta) - np.cos(2 * beta)) + (3 * u11 + u22) * np.cos(2 * alpha) * np.cos(2 * beta) + 2 * u22 * np.cos(4 * alpha)

    H = np.array([[V0, V1, V2], [np.conj(V1), V11, V12], [np.conj(V2), np.conj(V12), V22]])
    return np.moveaxis(H, (0, 1), (-2, -1))


def hamiltonian(model, p, k_x, k_y):
    alpha = 0.5 * k_x
    beta = (np.sqrt(3) / 2) * k_y
    return H_NN(alpha, beta, p) if model == "NN" else H_TNN(alpha, beta, p)

## SOC and the Zeeman Hamiltonian

Spin is conserved, so the $6\times6$ Hamiltonian is two $3\times3$ blocks and each block is diagonalized on its own. The field couples to the on-site orbital moment ($L_z$ mixes $d_{xy}$ and $d_{x^2-y^2}$) and to the spin,
$$H_{\uparrow} = H + \frac{\lambda}{2}L_z + \mu_B B\,L_z + \frac{g_s}{2}\mu_B B, \qquad H_{\downarrow} = H - \frac{\lambda}{2}L_z + \mu_B B\,L_z - \frac{g_s}{2}\mu_B B .$$
In each block the valence band is the lowest band (index 0) and the conduction bands are 1 and 2. We work at $T = 0$ with the chemical potential in the gap.

In [ ]:
L_z = np.array([[0, 0, 0], [0, 0, 2j], [0, -2j, 0]])


def zeeman(H, lam, B):
    up = H + (lam / 2) * L_z + mu_B * B * L_z + (g_s / 2) * mu_B * B * np.eye(3)
    dn = H - (lam / 2) * L_z + mu_B * B * L_z - (g_s / 2) * mu_B * B * np.eye(3)
    return up, dn

## k-space grids and masks

Every sum over $\mathbf{k}$ runs over a square grid with a mask. For the full Brillouin zone this is the grid `k_x = k_y = np.linspace(-5, 5, N)` with the hexagon mask `BZ`, and $dA = dk_x\,dk_y$. Near the band edge only the valleys matter, and there the disks are just another mask on a finer square grid: spacing $h$ around $K = (4\pi/3, 0)$ and $K' = -K$, with the mask $|\mathbf{k} - K| \le R$ and $dA = h^2$. Derivatives are `np.gradient` on the square grid. The disk grids carry two extra points beyond $R$, so the differences at the rim are central.

Both grids contain $-\mathbf{k}$ for every $\mathbf{k}$, so time reversal pairs every point exactly. At $B = 0$ each valley gives a term much larger than the field-induced signal (3 to 300 times, depending on the photon energy), and the two valleys have to cancel, so the pairing must be exact.

A grid is a tuple `(k_x, k_y, mask)`. `bz_grid` gives one (the hexagon), `valley_grids` gives two (the disks at $K$ and $K'$), and `eta_mag` loops over whichever list it gets.

The production code takes derivatives pointwise with a step of $10^{-4}/a$ instead. The two agree to $O(h^2)$, except near the anticrossings of the two conduction bands, where the velocity-shift term is not converged on either grid (the WSe$_2$ map around 3.0-3.5 eV).

In [ ]:
K_len = 4 * np.pi / 3
theta = np.arange(7) * np.pi / 3
hex_x, hex_y = K_len * np.cos(theta), K_len * np.sin(theta)


def bz_grid(N):
    k_x = np.linspace(-5, 5, N)
    k_x = 0.5 * (k_x - k_x[::-1])      # linspace is off by ~1e-15; this makes -k exactly a grid point
    k_xv, k_yv = np.meshgrid(k_x, k_x)
    BZ = (
        (np.abs(k_yv) < K_len * np.sqrt(3) / 2) &
        (np.abs(k_yv) < -np.sqrt(3) * np.abs(k_xv) + np.sqrt(3) * K_len)
    )
    return [(k_x, k_x, BZ)]


def disk(R, h):
    n = int(np.floor(R / h + 1e-9))
    q = h * np.arange(-n - 2, n + 3)      # two extra points beyond R on each side
    q_xv, q_yv = np.meshgrid(q, q)
    return q, q_xv**2 + q_yv**2 <= R**2 + 1e-12      # keep the rim points at |q| = R


def valley_grids(R, h):
    q, mask = disk(R, h)
    return [(K_len + q, q, mask), (-K_len + q, q, mask)]

## Bands and velocity operators

In one block, `eigh` gives $E_n$ and $|u_n\rangle$, and $v^a_{nm} = \langle u_n|\partial_a H|u_m\rangle$ with $\partial_a H$ from `np.gradient`.

In [ ]:
def bands(H, dk):
    E, V = np.linalg.eigh(H)
    dH_dkx = np.gradient(H, dk, axis=1)
    dH_dky = np.gradient(H, dk, axis=0)
    v_kx = np.einsum('ijkn, ijkl, ijlm->ijnm', V.conj(), dH_dkx, V)
    v_ky = np.einsum('ijkn, ijkl, ijlm->ijnm', V.conj(), dH_dky, V)
    return E, V, v_kx, v_ky

## Berry curvature and orbital moment

$$\Omega_n = -2\,\mathrm{Im}\sum_{m\ne n}\frac{v^x_{nm}v^y_{mn}}{(E_n - E_m)^2}, \qquad M_n = -\kappa_{\rm orb}\,2\,\mathrm{Im}\sum_{m\ne n}\frac{v^x_{nm}v^y_{mn}}{E_n - E_m}.$$
$\Omega_n$ is in units of $a^2$ and $M_n$ in Bohr magnetons, with $\kappa_{\rm orb} = m_e a^2/\hbar^2$ per eV ($1.336$ eV$^{-1}$ for MoS$_2$).

In [ ]:
def berry_and_moment(E, v_kx, v_ky, a):
    nb = E.shape[-1]
    dE = E[..., :, None] - E[..., None, :]
    dE[..., np.arange(nb), np.arange(nb)] = np.inf
    v_xy = v_kx * np.einsum('ijnm->ijmn', v_ky)      # v^x_nm v^y_mn

    Omega = -2 * (v_xy / dE**2).sum(axis=-1).imag
    kappa_orb = m_e * (a * 1e-10)**2 / hbar**2 * e
    M = -kappa_orb * 2 * (v_xy / dE).sum(axis=-1).imag
    return Omega, M

## Quantum metric

Interband metric between the valence band and conduction band $c$, with the bare energies (Zeeman in $H$, no moment shift),
$$g^{bc}_{cv} = \frac{\mathrm{Re}[v^b_{vc}v^c_{cv}]}{(E_c - E_v)^2}.$$

In [ ]:
def metric(E, v_kx, v_ky, c):
    v_vc = np.stack([v_kx[..., 0, c], v_ky[..., 0, c]], axis=-1)
    v_cv = np.stack([v_kx[..., c, 0], v_ky[..., c, 0]], axis=-1)
    dE = E[..., c] - E[..., 0]
    return np.einsum('ijb, ijc->ijbc', v_vc, v_cv).real / dE[..., None, None]**2

## Field term 1 -- energy shift

Besides the Zeeman Hamiltonian, the field acts on the itinerant orbital moment $M_n$ in three places. The first is the energy,
$$\varepsilon_n = E_n - \mu_B B\,M_n .$$
It is used only in the resonance, the argument of $\delta_\nu$. The kernel $g^{bc}_{cv}$ keeps the bare $E_c - E_v$.

In [ ]:
def energy_shift(E, M, B):
    return E - mu_B * B * M

## Field term 2 -- velocity shift

The same shift changes the group velocities that enter the velocity difference,
$$\Delta^a_{cv} = \left(\partial_a E_c - \mu_B B\,\partial_a M_c\right) - \left(\partial_a E_v - \mu_B B\,\partial_a M_v\right), \qquad \partial_a E_n = v^a_{nn},$$
with $\partial_a M_n$ again from `np.gradient`.

In [ ]:
def group_velocity(v_kx, v_ky, M, B, dk):
    dE_dkx = np.einsum('ijnn->ijn', v_kx).real - mu_B * B * np.gradient(M, dk, axis=1)
    dE_dky = np.einsum('ijnn->ijn', v_ky).real - mu_B * B * np.gradient(M, dk, axis=0)
    return dE_dkx, dE_dky

## Field term 3 -- phase-space factor

The Berry curvature changes the density of states in $\mathbf{k}$ space,
$$D_n = 1 + \frac{e a^2}{\hbar}\,B\,\Omega_n ,$$
and each transition is weighted by $D_v$ of the occupied band. For MoS$_2$, $e a^2/\hbar = 1.55\times10^{-4}$ T$^{-1}$.

In [ ]:
def phase_space(Omega, B, a):
    return 1 + e * (a * 1e-10)**2 / hbar * B * Omega

## Injection-current kernel

The magnetic injection current (Eq. 6 of the paper) is
$$\eta^{a;bc}(\omega) = -\frac{2\pi e^3 a}{\hbar^2}\int\frac{d^2k}{(2\pi)^2}\sum_{c} D_v\,\Delta^a_{cv}\,g^{bc}_{cv}\,\delta_\nu(\varepsilon_{cv} - \hbar\omega), \qquad \delta_\nu(x) = \frac{\nu/\pi}{x^2 + \nu^2}.$$
In code units ($e = \hbar = a = 1$, energies in eV)
$$\eta_{\rm code} = \pi\,\frac{dA}{(2\pi)^2}\sum_{\mathbf{k}}\sum_{\rm blocks}\sum_{c = 1,2} D_v\,\Delta^a_{cv}\,g^{bc}_{cv}\,\delta_\nu(\hbar\omega - \varepsilon_{cv}).$$
The Lorentzian is summed exactly over every transition. The tensor is symmetrized in $b \leftrightarrow c$ (linear polarization) and averaged over the three $C_3$ rotations, and the current in A m V$^{-2}$ s$^{-1}$ is $\eta = -2\,(e^3 a/\hbar^2)\,\eta_{\rm code}$. The figures show $\eta^{x;xx}$.

In [ ]:
def injection_terms(E, v_kx, v_ky, dE_dkx, dE_dky, E_res, D_v):
    eps, T = [], []
    for c in range(1, E.shape[-1]):
        Delta = np.stack([dE_dkx[..., c] - dE_dkx[..., 0], dE_dky[..., c] - dE_dky[..., 0]], axis=-1)
        g = metric(E, v_kx, v_ky, c)
        T.append(np.einsum('ij, ija, ijbc->ijabc', D_v, Delta, g))
        eps.append(E_res[..., c] - E_res[..., 0])
    return np.stack(eps, axis=-1), np.stack(T, axis=2)


def lorentz_sum(omega, nu, eps, T):
    # exact sum over every transition, in chunks so the Lorentzian matrix stays small
    eps = eps.ravel()
    T = T.reshape(len(eps), -1)
    out = np.zeros((len(omega), T.shape[1]))
    chunk = 4_000_000 // len(omega) + 1      # about 4e6 Lorentzian entries (32 MB) at a time
    for i in range(0, len(eps), chunk):
        lor = (nu / np.pi) / ((omega[:, None] - eps[None, i:i + chunk])**2 + nu**2)
        out += lor @ T[i:i + chunk]
    return out


def C3(T):
    out = 0
    for n in range(3):
        c, s = np.cos(2 * np.pi * n / 3), np.sin(2 * np.pi * n / 3)
        R = np.array([[c, -s], [s, c]])
        out = out + np.einsum('ia, jb, kc, ...abc->...ijk', R, R, R, T)
    return out / 3


def to_tensor(acc, dA):
    T = dA / (2 * np.pi)**2 * acc.reshape(-1, 2, 2, 2)
    return C3(0.5 * (T + np.swapaxes(T, -1, -2)))


def eta_SI(a):
    return -2 * e**3 * (a * 1e-10) / hbar**2      # code units -> A m V^-2 s^-1

## Shift current

Only Fig. 5 and the k.p model need it. It uses the projector form, $P_n = |u_n\rangle\langle u_n|$ with `np.gradient` derivatives,
$$C^{mn}_{a;bc} = \mathrm{tr}\left[P_n\,\partial_b P_m\,\partial_a\partial_c P_n\right] + \mathrm{tr}\left[P_n\,\partial_b P_m\,\partial_a P_m\,\partial_c P_n\right],$$
$\bar C^{mn} = (C^{mn} - C^{nm})/2$, symmetrized in $b \leftrightarrow c$, and
$$\sigma^{a;(bc)}_{\rm shift} = -2\pi\,\frac{dA}{(2\pi)^2}\sum_{\mathbf{k}}\sum_c D_v\,\mathrm{Im}\,\bar C^{cv}_{a;(bc)}\,\delta_\nu(\hbar\omega - \varepsilon_{cv})$$
in code units, times $e^2 a/\hbar$ for the sheet conductivity.

In [ ]:
def shift_terms(V, dk):
    P = np.einsum('ijkn, ijln->ijnkl', V, V.conj())
    dP = np.stack([np.gradient(P, dk, axis=1), np.gradient(P, dk, axis=0)])
    ddP = np.stack([np.stack([np.gradient(dP[0], dk, axis=1), np.gradient(dP[0], dk, axis=0)]),
                    np.stack([np.gradient(dP[1], dk, axis=1), np.gradient(dP[1], dk, axis=0)])])

    hess = np.einsum('ijnpq, bijmqr, ayijnrp -> ijmnaby', P, dP, ddP, optimize='optimal')
    cube = np.einsum('ijnpq, bijmqr, aijmrs, yijnsp -> ijmnaby', P, dP, dP, dP, optimize='optimal')
    C_mn = hess + cube
    C_asym = 0.5 * (C_mn - np.einsum('ijmnaby -> ijnmaby', C_mn))
    C_sym_by = 0.5 * (C_asym + np.einsum('ijmnaby -> ijmnayb', C_asym))
    return C_sym_by[:, :, 1:, 0].imag      # m = c (bands 1, 2), n = v (band 0)


def shift_SI(a):
    return e**2 * (a * 1e-10) / hbar / 1e-15       # code units -> nm uA V^-2

## Putting it together

`eta_mag` goes through each grid in strips of rows. Each strip has a 2-row halo on both sides, so `np.gradient` (also the second one, for $\partial_a M$) is central at every point inside the mask, and memory stays small on the fine disk grids. `terms` picks the field terms: `"E"` energy shift, `"V"` velocity shift, `"D"` phase-space factor. `"EVD"` (all three) is used for every tight-binding figure from Fig. 7 on, `""` is the Zeeman Hamiltonian alone.

In [ ]:
def eta_mag(mat, model, B, omega, nu, grids, func="GGA", lam=None, terms="EVD", shift=False, rows=200):
    p = params(mat, func, model)
    lam = lambda_SOC[mat] if lam is None else lam

    acc_eta, acc_shift = 0, 0
    for k_x, k_y, mask in grids:
        dk = k_x[1] - k_x[0]
        for start in range(0, len(k_y), rows):
            stop = min(start + rows, len(k_y))
            lo, hi = max(start - 2, 0), min(stop + 2, len(k_y))
            inside = np.zeros((hi - lo, len(k_x)), bool)
            inside[start - lo:stop - lo] = mask[start:stop]
            if not inside.any():
                continue
            k_xv, k_yv = np.meshgrid(k_x, k_y[lo:hi])

            for H in zeeman(hamiltonian(model, p, k_xv, k_yv), lam, B):
                E, V, v_kx, v_ky = bands(H, dk)
                Omega, M = berry_and_moment(E, v_kx, v_ky, p["a"])

                E_res = energy_shift(E, M, B if "E" in terms else 0)
                dE_dkx, dE_dky = group_velocity(v_kx, v_ky, M, B if "V" in terms else 0, dk)
                D_v = phase_space(Omega[..., 0], B if "D" in terms else 0, p["a"])

                eps, T = injection_terms(E, v_kx, v_ky, dE_dkx, dE_dky, E_res, D_v)
                acc_eta = acc_eta + lorentz_sum(omega, nu, eps[inside], T[inside])
                if shift:
                    S = D_v[..., None, None, None, None] * shift_terms(V, dk)
                    acc_shift = acc_shift + lorentz_sum(omega, nu, eps[inside], S[inside])

    eta = to_tensor(np.pi * acc_eta, dk**2)
    if shift:
        return eta, to_tensor(-2 * np.pi * acc_shift, dk**2)
    return eta

## Band gap and band width

$E_g$ is the smallest direct gap on a fine line through $K$ at $B = 0$ (both blocks), and $E_{\max}$ the largest interband energy $E_2 - E_0$ over the zone.

In [ ]:
def gap(mat, model="TNN", func="GGA", lam=None):
    p = params(mat, func, model)
    lam = lambda_SOC[mat] if lam is None else lam
    q = np.linspace(-0.05, 0.05, 401)
    H = hamiltonian(model, p, K_len + q, np.zeros_like(q))
    gaps = []
    for H_s in zeeman(H, lam, 0):
        E = np.linalg.eigvalsh(H_s)
        gaps.append((E[:, 1] - E[:, 0]).min())
    return min(gaps)


def E_max(mat, model="TNN", func="GGA"):
    p = params(mat, func, model)
    k = np.linspace(-2 * np.pi, 2 * np.pi, 401)      # covers the whole zone
    k_xv, k_yv = np.meshgrid(k, k)
    H = hamiltonian(model, p, k_xv, k_yv)
    widths = []
    for H_s in zeeman(H, lambda_SOC[mat], 0):
        E = np.linalg.eigvalsh(H_s)
        widths.append((E[..., 2] - E[..., 0]).max())
    return max(widths)

## Löwdin reduction at $K$

Near $K$ ($\tau = +1$, spin-up block, which holds the band edge) we rotate to $|c\rangle = |d_{z^2}\rangle$, $|v\rangle = (|d_{x^2-y^2}\rangle + i|d_{xy}\rangle)/\sqrt2$ and the remote $|u\rangle = (|d_{x^2-y^2}\rangle - i|d_{xy}\rangle)/\sqrt2$. $L_z$ is diagonal in this basis, so the spin-orbit coupling only moves the energies at $K$, $E_v = E_v^0 + \lambda$ and $E_u = E_u^0 - \lambda$. To second order,
$$H^{\rm eff}_{ij} = H_{ij} + \frac12\,H_{iu}H_{uj}\left[\frac{1}{E_i - E_u} + \frac{1}{E_j - E_u}\right],$$
and fitting $H^{\rm eff}_{cv}(\mathbf{q}) = t\,(q_x - iq_y) + \kappa\,(q_x + iq_y)^2 + \dots$ on small rings around $K$ gives $v = at$ and the trigonal warping $\kappa$ (in eV Å$^2$ after multiplying by $a^2$). Fitting the diagonal elements to $H^{\rm eff}_{cc} = E_c + \gamma_c q^2 + \dots$ and $H^{\rm eff}_{vv} = E_v + \gamma_v q^2 + \dots$ gives $w = (\gamma_c - \gamma_v)/2$ (times $a^2$ for eV Å$^2$) and $m_D = (E_c - E_v)/2$.

`warping` returns $\kappa$ in eV Å$^2$. `kappa_NN_closed` is the NN closed form (Eq. 11 of the paper), evaluated with the spin-orbit shifts in the energies ($E_v + \lambda$, $E_u - \lambda$) for the band-edge spin block, as a check.

In [ ]:
def lowdin(mat, func="GGA", model="TNN", lam=None):
    p = params(mat, func, model)
    lam = lambda_SOC[mat] if lam is None else lam
    c = np.array([1, 0, 0])
    v = np.array([0, 1j, 1]) / np.sqrt(2)
    u = np.array([0, -1j, 1]) / np.sqrt(2)
    U = np.stack([c, v, u], axis=1)

    r = np.repeat([0.002, 0.004, 0.006, 0.01], 64)
    phi = np.tile(np.linspace(0, 2 * np.pi, 64, endpoint=False), 4)
    q_x, q_y = r * np.cos(phi), r * np.sin(phi)

    H_up_K = zeeman(hamiltonian(model, p, np.array([K_len]), np.array([0.0])), lam, 0)[0]
    H_K = U.conj().T @ H_up_K[0] @ U
    E_c, E_v, E_u = H_K.diagonal().real
    H = U.conj().T @ zeeman(hamiltonian(model, p, K_len + q_x, q_y), lam, 0)[0] @ U

    H_cv = H[:, 0, 1] + 0.5 * H[:, 0, 2] * H[:, 2, 1] * (1 / (E_c - E_u) + 1 / (E_v - E_u))
    H_cc = H[:, 0, 0].real + np.abs(H[:, 0, 2])**2 / (E_c - E_u)
    H_vv = H[:, 1, 1].real + np.abs(H[:, 1, 2])**2 / (E_v - E_u)

    z = q_x - 1j * q_y
    # t (q_x - i q_y) + kappa (q_x + i q_y)^2, plus the next allowed terms so the two we keep are not biased
    A = np.stack([z, z.conj()**2, r**2 * z, z.conj(), z**2], axis=1)
    coef = np.linalg.lstsq(A, H_cv, rcond=None)[0]
    # E + gamma q^2 + trigonal q^3 + q^4
    X = np.stack([np.ones_like(r), r**2, q_x * (q_x**2 - 3 * q_y**2), r**4], axis=1)
    coef_c = np.linalg.lstsq(X, H_cc, rcond=None)[0]
    coef_v = np.linalg.lstsq(X, H_vv, rcond=None)[0]
    return dict(v=coef[0].real, kappa=coef[1].real, w=(coef_c[1] - coef_v[1]) / 2, mD=(E_c - E_v) / 2, lam=lam, a=p["a"])


def warping(mat, func="GGA", model="TNN"):
    r = lowdin(mat, func, model)
    return r["kappa"] * r["a"]**2      # eV A^2


def kappa_NN_closed(mat, func="GGA"):
    p = params(mat, func, "NN")
    lam = lambda_SOC[mat]
    t1, t2, t11, t12, t22 = p["t1"], p["t2"], p["t11"], p["t12"], p["t22"]
    E_c = p["epsilon1"] - 3 * p["t0"]
    E_v = p["epsilon2"] - 1.5 * (t11 + t22) - 3 * np.sqrt(3) * t12 + lam
    E_u = p["epsilon2"] - 1.5 * (t11 + t22) + 3 * np.sqrt(3) * t12 - lam
    gamma3 = 3 * np.sqrt(2) / 8 * (t2 - np.sqrt(3) * t1) + 9 * np.sqrt(6) / 16 * (t1 - np.sqrt(3) * t2) * (t11 - t22) * (1 / (E_c - E_u) + 1 / (E_v - E_u))
    return p["a"]**2 * gamma3 / 2

## Two-band k.p model

The warped Dirac model, for valley $\tau$ and spin $s$,
$$H = \mathbf{d}\cdot\boldsymbol{\sigma}, \quad d_x = v\tau k_x + \kappa(k_x^2 - k_y^2), \quad d_y = v k_y - 2\kappa\tau k_x k_y, \quad d_z = m_D + w k^2 - \tau(b + s\lambda/2),$$
with $b = \mu_B B$, summed over the spin-valley locked pair $(\tau, s) = (+,+)$ and $(-,-)$. The field only enters through $b$ in $d_z$. The kernels are the same as for the tight-binding model, on the square grid `linspace(-R, R, N_k)` with the mask $k^2 \le R^2$.

The first-principles parameters are those of Kormányos *et al.*, 2D Mater. **2**, 022001 (2015), for the band-edge spin pair, with $k$ in 1/Å: $m_D = (E_A + \lambda)/2$ and $w = (\hbar^2/4m_e)(1/m_c + 1/|m_v|) - v^2/E_A$. `kp_chi` gives their threshold coefficient $\chi = \eta^{x;xx}(30\,{\rm T})/30$ T at $E_A + 0.01$ eV (disk of radius 0.2/Å, $\nu = 5$ meV) and saves it, so `edge.ipynb` can load it.

In [ ]:
def kp_hamiltonian(q_xv, q_yv, tau, s, b, kp):
    d_x = kp["v"] * tau * q_xv + kp["kappa"] * (q_xv**2 - q_yv**2)
    d_y = kp["v"] * q_yv - 2 * kp["kappa"] * tau * q_xv * q_yv
    d_z = kp["mD"] + kp["w"] * (q_xv**2 + q_yv**2) - tau * (b + s * kp["lam"] / 2)
    H = np.array([[d_z, d_x - 1j * d_y], [d_x + 1j * d_y, -d_z]])
    return np.moveaxis(H, (0, 1), (-2, -1))


def kp_currents(kp, b, omega, nu, R, N_k, pairs=((1, 1), (-1, -1)), shift=False):
    q, mask = disk(R, 2 * R / (N_k - 1))      # spacing of linspace(-R, R, N_k)
    dk = q[1] - q[0]
    q_xv, q_yv = np.meshgrid(q, q)
    acc_eta, acc_shift = 0, 0
    for tau, s in pairs:
        E, V, v_kx, v_ky = bands(kp_hamiltonian(q_xv, q_yv, tau, s, b, kp), dk)
        dE_dkx = np.einsum('ijnn->ijn', v_kx).real
        dE_dky = np.einsum('ijnn->ijn', v_ky).real
        eps, T = injection_terms(E, v_kx, v_ky, dE_dkx, dE_dky, E, np.ones_like(E[..., 0]))
        acc_eta = acc_eta + lorentz_sum(omega, nu, eps[mask], T[mask])
        if shift:
            acc_shift = acc_shift + lorentz_sum(omega, nu, eps[mask], shift_terms(V, dk)[mask])
    eta = to_tensor(np.pi * acc_eta, dk**2)
    if shift:
        return eta, to_tensor(-2 * np.pi * acc_shift, dk**2)
    return eta


#           E_A    v      kappa  m_c    m_v     (eV, eV A, eV A^2, m_e)
korm = {"MoS2":  (1.670, 2.76, -1.45, 0.43, -0.54),
        "MoSe2": (1.400, 2.53, -1.23, 0.49, -0.59),
        "WS2":   (1.632, 3.34, -1.09, 0.26, -0.35),
        "WSe2":  (1.337, 3.17, -0.93, 0.28, -0.36)}
hbar2_2m = 3.80998     # hbar^2 / 2 m_e in eV A^2


def kp_first_principles(mat):
    E_A, v, kap, m_c, m_v = korm[mat]
    lam = lambda_SOC[mat]
    return dict(v=v, kappa=kap, w=0.5 * hbar2_2m * (1 / m_c + 1 / abs(m_v)) - v**2 / E_A, mD=(E_A + lam) / 2, lam=lam)


def kp_chi(N_k):
    fn = f"data/kp_chi_N{N_k}.npz"
    if not os.path.exists(fn):
        chi = []
        for m in materials:
            omega = np.array([korm[m][0] + 0.01])
            eta = kp_currents(kp_first_principles(m), mu_B * 30, omega, 0.005, 0.2, N_k)
            chi.append(eta_SI(1.0) * eta[0, 0, 0, 0] / 30)      # k in 1/A, so a = 1 A
        np.savez(fn, chi=chi)
    return dict(zip(materials, np.load(fn)["chi"]))